# INTRODUCTION TO THE GAME

In this notebook we are using the python package "millionaire-client" to interact with the deployed application for the NLP assignment 2026.

### Game interaction

In [ ]:
from google.colab import drive
import os
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
import sys
import os

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/NLP/NLP_assignment_api_client'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/gdrive/MyDrive/NLP/NLP_assignment_api_client', '/content/gdrive/MyDrive/NLP_assignment_api_client']


In [ ]:
from millionaire_client import MillionaireClient, AuthenticationError

In [ ]:
from google.colab import userdata
pwd = userdata.get('poli-millionaire')

In [ ]:
API_URL = "http://131.175.15.22:51111/"
username = "binazz"
password = pwd

In [ ]:
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")


Welcome, binazz! (Role: student)


The game counts 6 different categories, each of them having 15 question. It's possible to select between *text mode* and *speech mode*.

In [ ]:
# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")


=== Available Competitions ===
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)
  4: Philosophy and Psychology (15 questions)
  5: News (15 questions)


#DEFINING APPROACH

Questions about entertainment, ancient history and politics, science and nature, philosophy and psychology are answered using a RAG approach.

Since search engines suffer from **Keyboard Diluition**, we are going to use an LLM as an "Agentic Sniper". Once the model output the best search keywords, those are queried into the chosen search engines. We are going to retrieve the needed information from Wikipedia for all those cathegories, except for the news one, when we are going to use an Guardian API.

The LLM model used in perform such tasks is Mistral 7B with 4-bit quantization.


The math category will use a different approach instead.

### Model download

In [ ]:
!pip install transformers accelerate bitsandbytes
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# 4-bit config to maximize available VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Let accelerate handle the device map automatically
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

##LIMITS



*   **Hardware**: the inference runs on **Google Colab's free-tier T4** (16 GB VRAM). Since this doesn't allow to run the full precision Mistral-7B-Instruct-v0.3, we are going to use the 4-bit Quantization version.
*   **30s timeout**: this limits expecially the math category.



# RAG ARCHITECTURE


## WIKIPEDIA SNIPER

While generic web search engines return a high volume of unstructured, noisy data, trivia questions require highly factual, encyclopedic answers. To maximize efficiency under the strict 30-second game clock, we engineered the **Wikipedia Sniper**.

Instead of routing our queries through a third-party search engine, this pipeline connects directly to Wikipedia's native `Special:Search` endpoint. By strictly restricting our crosshairs to a curated encyclopedia, the Sniper drastically improves our signal-to-noise ratio. This targeted approach allows the engine to resolve the exact article URL, download the raw HTML, and heuristically extract the specific paragraph containing the answer with maximum precision and minimal latency.

The pipeline operates in four distinct phases:

1. **Intelligent Query Planning**: Rather than basic string matching, Mistral analyzes the trivia question to extract its semantic core, generating a highly focused set of search keywords.

2. **Direct Data Retrieval**: The system navigates to the resolved article URL and deploys BeautifulSoup to download the raw HTML, stripping away web formatting to extract purely the textual <p> tags.
3. **Heuristic Context Compression**: Feeding an entire article to an LLM causes context overflow. We built an algorithm that scores every paragraph based on its word overlap with the question and multiple-choice options. Only the Top 3 highest-scoring paragraphs are kept, compressing thousands of words into a hyper-dense context window.
    - **The Search Matrix**: The system builds a massive array of "Search Words" by combining: 1) The LLM's initial keywords, 2) Every significant noun from the question, and 3) Every specific entity from the four multiple-choice options.
    - **Paragraph Scoring**: The algorithm scans every paragraph in the dowload article, assigning a mathematical score based on how many "Search Words" appear within that specific text block.
    - **Extraction**: The paragraphs are sorted by score, and only the Top 3 highest-scoring matches are selected. This compresses thousands of words into a hyper-dense, highly relevant context window.
  4. **Semantic Telemetry**: Before answering, the system cross-references the extracted context against the options. If it detects a high overlap ($\ge$ 2 matches), it logs a "Semantic Match," confirming the pipeline has isolated the exact factual answer before hitting the generation phase.

First, let's download the necessary libraries.

In [ ]:
!pip install beautifulsoup4 requests -q

Let's now define the setup.

In [ ]:
import requests
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In order to answer `comp_id in [0,1,2,4]`, two different specialized functions are used:
- `get_direct_wiki_context` (`comp_id in [0, 1]`): Designed for general trivia where a single source of truth is usually sufficient. Mistral is prompted to identify a single core entity (e.g., "Audrey Hepburn") and separate it from secondary search keywords using a pipe character (|). To maximize speed, only the #1 top Wikipedia result is downloaded and parsed.
- `get_direct_wiki_context_detailed` (`comp_id in [2, 4]`): Designed for highly specific or nuanced categories. Mistral is prompted with a "Reasoning" template to deduce the context and generate 1 to 3 highly targeted Wikipedia search terms, strictly banning generic WH-words. The engine combines these queries to force a highly specific search, downloads the Top 2 results from Wikipedia, and pools their text. The Smart Scroller then selects the Top 3 paragraphs across both documents. This added reasoning step ensures the engine retrieves the correct pages even for complex queries.

In [ ]:
def get_direct_wiki_context(question_text, options, model, tokenizer, comp_id):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")

        print(f"   (category {comp_id} detected: Using Entity Pipe-Split)")
        prompt = """[INST] You are an expert Wikipedia search assistant.
        Identify the core subject of the trivia question (person, movie, event).
        CRITICAL RULE: Always deduce and output the FULL, SPECIFIC NAME of the subject. If the question uses a last name like "Armstrong" or says "the film", use context to output the exact full name (e.g., "Louis Armstrong").
        You MUST output the full core subject, followed by a pipe character '|', followed by 2 or 3 crucial keywords from the question.

        Example 1:
        Question: What is the fundamental principle of M3GAN, the artificial intelligence doll in the film?
        Output: M3GAN | artificial intelligence doll plot

        Example 2:
        Question: What was Audrey Hepburn's maiden name before she married Mel Ferrer?
        Output: Audrey Hepburn | maiden name married Ferrer

        Question: {question_text}
        Output: [/INST]"""

        # Generate the Split Query
        inputs = tokenizer(prompt.format(question_text=question_text), return_tensors="pt").to("cuda")
        input_length = inputs.input_ids.shape[1]

        outputs = model.generate(**inputs, max_new_tokens=20, pad_token_id=tokenizer.eos_token_id)
        raw_output = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
        raw_output = raw_output.replace('"', '').replace("'", '').replace('\n', ' ')

        # Parse the Pipe Split
        if '|' in raw_output:
            wiki_query, scroller_keywords = raw_output.split('|', 1)
            wiki_query = wiki_query.strip()
            scroller_keywords = scroller_keywords.strip()
        else:
            # Fallback just in case Mistral forgets the pipe
            wiki_query = raw_output.strip()
            scroller_keywords = raw_output.strip()

        print(f"   (Agent querying Wikipedia for Page: '{wiki_query}')")
        print(f"   (Smart Scroller targeting Keywords: '{scroller_keywords}')")

        headers = {
            'User-Agent': 'MyTriviaBot/1.0 (contact: albana.lusha@mail.polimi.com)'
        }

        try:
            # Search Wikipedia using only the isolated Page Title
            search_url = "https://en.wikipedia.org/w/index.php"
            search_params = {'search': wiki_query, 'title': 'Special:Search', 'profile': 'advanced', 'fulltext': '1'}

            time.sleep(1.5)
            search_resp = requests.get(search_url, params=search_params, headers=headers, timeout=5)
            search_resp.raise_for_status()
            search_soup = BeautifulSoup(search_resp.text, 'html.parser')

            first_result = search_soup.find('div', class_='mw-search-result-heading')
            if not first_result:
                print("   (Direct Wikipedia search returned no pages.)")
                return ""

            article_link = "https://en.wikipedia.org" + first_result.find('a')['href']
            print(f"   (Found target page: {article_link})")

            # Download the full article
            article_resp = requests.get(article_link, headers=headers, timeout=5)
            article_soup = BeautifulSoup(article_resp.text, 'html.parser')
            paragraphs = article_soup.find_all('p')

            # SMART SCROLLING ALGORITHM: Full-Spectrum Text Magnet
            search_words = set((wiki_query + " " + scroller_keywords).lower().split())

            # Expanded stop words for trivia questions
            stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'with', 'by', 'about', 'as', 'through', 'of', 'is', 'was', 'her', 'she', 'he', 'his', 'it', 'that', 'from', 'what', 'which', 'who', 'how', 'why', 'where', 'when', 'did', 'does', 'do', 'are'}

            # 1. Inject the raw Question words! (Guarantees we never lose specific nouns like "stairs")
            question_words = set(re.findall(r'\b\w+\b', question_text.lower())) - stop_words
            search_words.update(question_words)

            # 2. Inject the Options words!
            for opt in options:
                opt_words = set(re.findall(r'\b\w+\b', opt.text.lower())) - stop_words
                search_words.update(opt_words)

            scored_paragraphs = []

            for p in paragraphs:
                text = p.text.strip()
                if len(text) < 50:
                    continue

                text_lower = text.lower()
                # Give a point for every search keyword found in this paragraph
                score = sum(1 for word in search_words if word in text_lower)
                scored_paragraphs.append((score, text))

            # Sort paragraphs by score (highest first)
            scored_paragraphs.sort(key=lambda x: x[0], reverse=True)

            # Build context from ONLY the top 3 highest-scoring paragraphs
            context = ""
            for score, text in scored_paragraphs[:3]:
                if score > 0: # Only include it if it's relevant!
                    context += text + " "

            # Fallback: If no paragraphs matched the keywords, just take the intro
            if not context:
                print("   (Keywords not found deep in text. Falling back to introduction.)")
                for p in paragraphs[:3]:
                    if len(p.text.strip()) > 50:
                        context += p.text.strip() + " "

            # Semantic Telemetry Check
            stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'with', 'by', 'about', 'as', 'through', 'of'}
            highest_overlap = 0
            best_option = None
            context_words = set(re.findall(r'\b\w+\b', context.lower()))

            for opt in options:
                opt_words = set(re.findall(r'\b\w+\b', opt.text.lower())) - stop_words
                if not opt_words: continue
                overlap_count = len(opt_words.intersection(context_words))
                if overlap_count > highest_overlap:
                    highest_overlap = overlap_count
                    best_option = opt.id

            if highest_overlap >= 2:
                print(f"   (Semantic Match! High overlap detected for Option [{best_option}].)")
            else:
                print("   (Context retrieved, but semantic overlap with options is low.)")

            return context.strip()

        except Exception as e:
            print(f"   (Direct Wikipedia Error: {e})")
            return ""

In [ ]:
def get_direct_wiki_context_detailed(question_text, options, model, tokenizer, comp_id):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")

        # =====================================================
        # DYNAMIC AI SNIPER ROUTING (Mistral Reasoning Edition)
        # =====================================================
        planner_prompt = f"""You are an elite Wikipedia Search Planner.
        Read the trivia question, briefly reason about the core subject, and then provide 1 to 3 highly specific Wikipedia search terms.

        CRITICAL RULES:
        1. BAN QUESTION WORDS: Never use Who, What, Where, When, Why, How.
        2. EXTRACT NOUNS: Focus on the historical event, person, or scientific concept.

        EXAMPLE 1:
        Question: In what year did the ocean liner Titanic sink in the Atlantic?
        Reasoning: The question is asking for the date of a specific historical maritime disaster.
        KEYWORDS: Sinking of the Titanic, RMS Titanic

        EXAMPLE 2:
        Question: What term describes the act of separating people based on racial grounds without a reasonable justification?
        Reasoning: The user is looking for the sociological or legal concept of racial separation.
        KEYWORDS: Racial segregation, Discrimination

        Now, do the following question:
        Question: {question_text}
        Reasoning:"""

        # 1. Format and Generate with Mistral
        messages = [{"role": "user", "content": planner_prompt}]

        # Safely render the chat template to a string first
        prompt_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

        # Tokenize the string into a dictionary of tensors
        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

        # Extract the length specifically from the 'input_ids' tensor inside the dictionary
        input_length = inputs["input_ids"].shape[1]

        # Use **inputs to unpack the dictionary correctly
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            pad_token_id=tokenizer.eos_token_id,
            temperature=0.1
        )

        raw_response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

        # 2. Parse the Mistral Output
        try:
            keyword_string = raw_response.split("KEYWORDS:")[1].strip()
            queries = [kw.strip() for kw in keyword_string.split(",")]
        except IndexError:
            print("   (Mistral formatting failed. Using fallback noun extraction.)")
            words = [w for w in question_text.replace('?','').split() if len(w) > 5]
            queries = words[:3] if words else [question_text.split()[0]]

        # 3. Map to your existing Scroller Architecture
        wiki_query = queries[0] if queries else question_text.split()[0]
        scroller_keywords = " ".join(queries[1:]) if len(queries) > 1 else wiki_query

        print(f"   (Agent extracted base concepts: '{wiki_query}' | '{scroller_keywords}')")

        headers = {
            'User-Agent': 'MyTriviaBot/1.0 (contact: albana.lusha@mail.polimi.com)'
        }

        try:
            # ==============================
            # THE MULTI-PAGE DRAGNET UPGRADE
            # ==============================

            # ENHANCED QUERY: Combine the top 2 queries to force Wikipedia to be specific
            enhanced_query = " ".join(queries[:2]) if len(queries) > 1 else wiki_query
            print(f"   (Enhanced Wikipedia Search: '{enhanced_query}')")

            search_url = "https://en.wikipedia.org/w/index.php"
            search_params = {'search': enhanced_query, 'title': 'Special:Search', 'profile': 'advanced', 'fulltext': '1'}

            search_resp = requests.get(search_url, params=search_params, headers=headers, timeout=5)
            search_resp.raise_for_status()
            search_soup = BeautifulSoup(search_resp.text, 'html.parser')

            # MULTI-PAGE DRAGNET: Grab the top 2 results instead of just 1
            search_results = search_soup.find_all('div', class_='mw-search-result-heading', limit=2)

            if not search_results:
                print("   (Direct Wikipedia search returned no pages.)")
                return ""

            paragraphs = []

            # Loop through both top pages and pool their text together
            for result in search_results:
                article_link = "https://en.wikipedia.org" + result.find('a')['href']
                print(f"   (Fetching target page: {article_link})")

                try:
                    article_resp = requests.get(article_link, headers=headers, timeout=5)
                    article_soup = BeautifulSoup(article_resp.text, 'html.parser')
                    # Add all paragraphs from this page to our master pool
                    paragraphs.extend(article_soup.find_all('p'))
                except Exception as e:
                    print(f"   (Failed to fetch {article_link}: {e})")

            # =========================
            # SMART SCROLLING ALGORITHM
            # =========================

            search_words = set((wiki_query + " " + scroller_keywords).lower().split())

            # Expanded stop words for trivia questions
            stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'with', 'by', 'about', 'as', 'through', 'of', 'is', 'was', 'her', 'she', 'he', 'his', 'it', 'that', 'from', 'what', 'which', 'who', 'how', 'why', 'where', 'when', 'did', 'does', 'do', 'are'}

            # 1. Inject the raw Question words!
            question_words = set(re.findall(r'\b\w+\b', question_text.lower())) - stop_words
            search_words.update(question_words)

            # 2. Inject the Options words!
            for opt in options:
                opt_words = set(re.findall(r'\b\w+\b', opt.text.lower())) - stop_words
                search_words.update(opt_words)

            scored_paragraphs = []

            for p in paragraphs:
                text = p.text.strip()
                if len(text) < 50:
                    continue

                text_lower = text.lower()
                # Give a point for every search keyword found in this paragraph
                score = sum(1 for word in search_words if word in text_lower)
                scored_paragraphs.append((score, text))

            # Sort paragraphs by score (highest first)
            scored_paragraphs.sort(key=lambda x: x[0], reverse=True)

            # Build context from ONLY the top 3 highest-scoring paragraphs
            context = ""
            for score, text in scored_paragraphs[:3]:
                if score > 0: # Only include it if it's relevant!
                    context += text + " "

            # Fallback: If no paragraphs matched the keywords, just take the intro
            if not context:
                print("   (Keywords not found deep in text. Falling back to introduction.)")
                for p in paragraphs[:3]:
                    if len(p.text.strip()) > 50:
                        context += p.text.strip() + " "

            # Semantic Telemetry Check
            stop_words_telemetry = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'with', 'by', 'about', 'as', 'through', 'of'}
            highest_overlap = 0
            best_option = None
            context_words = set(re.findall(r'\b\w+\b', context.lower()))

            for opt in options:
                opt_words = set(re.findall(r'\b\w+\b', opt.text.lower())) - stop_words_telemetry
                if not opt_words: continue
                overlap_count = len(opt_words.intersection(context_words))
                if overlap_count > highest_overlap:
                    highest_overlap = overlap_count
                    best_option = opt.id

            if highest_overlap >= 2:
                print(f"   (Semantic Match! High overlap detected for Option [{best_option}].)")
            else:
                print("   (Context retrieved, but semantic overlap with options is low.)")

            return context.strip()

        except Exception as e:
            print(f"   (Direct Wikipedia Error: {e})")
            return ""

Once Mistral extracts the top paragraphs using the CrosseEncoder, it is asked to pick the right answer. This is performed using the `ask_llm_with_rag` function.

In [ ]:
import time
import re

def ask_llm_with_rag(question_text, options, context, model, tokenizer, search_time=0.0):
    # 1. Explicitly check and print if the search failed
    if not context:
        print("   (No external info found. Mistral is flying blind!)")

    print("   (Asking Mistral for the final answer...)")

    # Start the Inference Stopwatch!
    start_inference = time.time()

    # 2. Format the prompt
    prompt = "[INST] You are a highly intelligent trivia bot. Answer with ONLY the number of the correct option. Do not write anything else.\n\n"
    if context:
        prompt += f"Context: {context}\n\n"

    prompt += f"Question: {question_text}\n"
    for opt in options:
        prompt += f"[{opt.id}] {opt.text}\n"
    prompt += "Answer: [/INST]"

    # 3. Convert to tokens and send to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # 4. Generate answer (Using low temperature for factual strictness)
    input_length = inputs.input_ids.shape[1]
    outputs = model.generate(**inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id, temperature=0.1)

    raw_answer = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

    # Stop the Inference Stopwatch!
    inference_time = time.time() - start_inference
    total_time = search_time + inference_time

    # 5. Extract the number
    numbers = re.findall(r'\d+', raw_answer)
    if numbers:
        answer_id = int(numbers[0])
    else:
        print(f"   (AI didn't output a number! Raw output: '{raw_answer}'. Guessing Option 0...)")
        answer_id = options[0].id

    # 6. Print the beautiful Timing Report
    print("\n   === TIMING REPORT ===")
    print(f"   Search Time:    {search_time:.2f} seconds")
    print(f"   Inference Time: {inference_time:.2f} seconds")
    print(f"   TOTAL TIME:     {total_time:.2f} seconds")
    print("   ========================\n")

    return answer_id

## NEWS QUESTIONS

### Project Documentation: News Retrieval & Reasoning Engine

The system employs a multi-stage RAG architecture to answer trivia questions using real-time news data from **The Guardian API**, chosen for its structured JSON format, editorial integrity, and precise timestamping capabilities.


#### Pipeline Architecture

The retrieval process is split between two primary functions:

**1. Orchestration & Retrieval (`get_guardian_api_context`)**

* **The Surgical Planner:** A "blind" LLM extracts 2–3 high-signal keywords from the question, preventing "shortcut reasoning" by isolating keywords from answer options.
* **Temporal Filter:** The engine parses ISO dates from the question, ensuring the API only returns articles from the relevant timeframe.
* **Data Fetch:** The system retrieves four articles and consolidates them into a pool of individual paragraphs for scoring.

**2. Ranking & Extraction (`score_and_extract_news_context`)**
This function acts as a precision re-ranker to distill retrieved content:

* **Baseline Scoring:** Paragraphs are scored based on textual overlap with the trivia question to filter out irrelevant noise.
* **The Option Radar:** A semantic magnet that applies heavy multipliers (+10 for exact matches, +3 for partial matches) to paragraphs containing text from the options.
* **Narrative Preservation:** The system extracts the top-scoring paragraph and its successor to maintain contextual flow.

#### Verification & Telemetry

Before generating an answer, the system performs a **Semantic Telemetry Check**. It calculates the overlap between the retrieved context and the provided options; if the overlap is below a defined threshold, the retrieval is flagged as "low confidence" to prevent the model from answering based on insufficient data.

In [ ]:
import requests
import re
import warnings
import time
import torch
from datetime import datetime, timedelta

In [ ]:
GUARDIAN_KEY = "f1540f79-5160-44a7-a3b5-cc3f1c86313a"

def score_and_extract_news_context(article_text, question_text, options):
    """
    Scores paragraphs with a heavy multiplier for finding the exact options.
    Returns the winning paragraph + the one after it (capped at 1500 chars).
    """
    paragraphs = [p.strip() for p in article_text.split('\n') if len(p.strip()) > 50]
    if not paragraphs:
        return ""

    # 1. Base Question Words (Worth 1 point each)
    q_words = set(re.findall(r'\b\w+\b', question_text.lower()))
    stop_words = {
        'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'with', 'by', 'of', 'is', 'was', 'are',
        'what', 'who', 'when', 'where', 'why', 'how', 'which', 'did', 'does', 'describe', 'happen', 'happened', 'according'
    }
    q_words = q_words - stop_words

    # 2. Prepare Option Strings
    option_texts = [opt.text.lower() for opt in options]

    best_idx = -1
    highest_score = -1

    for idx, para in enumerate(paragraphs):
        para_lower = para.lower()

        # Add 1 point for every matching question word
        score = sum(1 for word in q_words if word in para_lower)

        # Heavy multipliers for finding the answers
        for opt_text in option_texts:
            if opt_text in para_lower:
                # If the exact option string is in the text, it is almost certainly the answer!
                score += 10
            else:
                # If not an exact match, check individual option words (worth 3 points each)
                # Useful for names like "Green Party" matching "Greens"
                opt_words = set(re.findall(r'\b\w+\b', opt_text)) - stop_words
                score += sum(3 for word in opt_words if word in para_lower)

        if score > highest_score:
            highest_score = score
            best_idx = idx

    # Grab the winning paragraph AND the next one!
    if best_idx != -1:
        best_context = paragraphs[best_idx]
        if best_idx + 1 < len(paragraphs):
            best_context += " " + paragraphs[best_idx + 1]

        # Generous 1500 char cap to prevent 30s timeouts
        return best_context[:1500].strip()

    return ""

In [ ]:
def get_guardian_api_context(question_text, options, model, tokenizer, comp_id):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")

        # ==========================================
        # PHASE 1: THE PROPER NOUN PLANNER
        # ==========================================
        planner_prompt = f"""You are a keyword extraction robot.
        Your goal is to extract exactly 2-3 search keywords.

        1. NO OPTIONS: Do not look at or mention the options.
        2. NO DATES: NEVER include years, months, days, or numbers.
        3. NO GENERIC: Never use "reported", "article", "according", "marketing", "price", "ranges".
        4. OUTPUT: ONLY 2-3 keywords on a single line.
        5. PRIORITIZE NAMES and nouns only.

        Question: {question_text}
        Output:"""

        # Format and Generate the query with Mistral
        messages = [{"role": "user", "content": planner_prompt}]
        prompt_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

        # Explicitly disable gradient tracking to protect VRAM
        with torch.no_grad():
            inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
            input_length = inputs["input_ids"].shape[1]

            outputs = model.generate(
                **inputs,
                max_new_tokens=15,  # Room to catch complete entity chains
                pad_token_id=tokenizer.eos_token_id,
                temperature=0.0,    # Sharp, deterministic extraction
                do_sample=False
            )

            raw_query = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

        # ==========================================
        # THE SANITIZATION WASH
        # ==========================================
        # 1. Lowercase everything for uniform evaluation
        clean_query = raw_query.lower()

        # 2. Obliterate logical operators, boolean traps, and structural conjunctions
        invalid_tokens = {'and', 'or', 'not', 'with', 'from', 'against', 'about', 'for', 'the', 'a', 'an', 'to', 'in', 'of', 'on', 'at'}
        query_words = [word for word in clean_query.split() if word not in invalid_tokens]

        # Hard-cap the list to exactly 3 words to physically protect the API
        query_words = query_words[:4]

        # 3. Purge destructive punctuation and special characters that disrupt API parsing
        query = " ".join(query_words)
        query = re.sub(r'[&"\'\(\)\,\.\?\!\-\+\:\;\/\#\*]', ' ', query)
        query = " ".join(query.split())  # Collapse duplicate whitespace

        print(f"   (Agent extracted news concepts: '{query}')")

        # Clear VRAM from the planner step immediately
        del inputs, outputs
        torch.cuda.empty_cache()

        # ==========================================
        # PHASE 2: TEMPORAL DATE-LOCKING
        # ==========================================
        date_match = re.search(r'\b\d{4}-\d{2}-\d{2}\b', question_text)
        if date_match:
            target_date = date_match.group(0)
            print(f"   (Target Date Strictly Locked: {target_date})")
        else:
            print("   (No ISO date found in question. Using live stream mode.)")
            target_date = None

        # ==========================================
        # PHASE 3: GUARDIAN API DIRECT FETCH
        # ==========================================
        # Uses your established global token variable
        search_url = "https://content.guardianapis.com/search"

        params = {
            'q': query,
            'api-key': "f1540f79-5160-44a7-a3b5-cc3f1c86313a",
            'show-fields': 'bodyText',
            'page-size': 4
        }

        if target_date:
            params['from-date'] = target_date
            params['to-date'] = target_date

        try:
            search_resp = requests.get(search_url, params=params, timeout=5)
            search_resp.raise_for_status()
            data = search_resp.json()

            results = data.get('response', {}).get('results', [])
            if not results:
                print("   (Guardian API returned no articles for this date/query window.)")
                return ""

            print(f"   (Guardian API Success: Extracted {len(results)} context blocks.)")

            # Master paragraph pool
            paragraphs = []
            for art in results:
                body_text = art.get('fields', {}).get('bodyText', '')
                art_paragraphs = [p.strip() for p in body_text.split('\n') if len(p.strip()) > 50]
                paragraphs.extend(art_paragraphs)

            # ==========================================
            # PHASE 4: OPTION-SCORING RADAR
            # ==========================================
            search_words = set(query.lower().split())
            stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'with', 'by', 'about', 'as', 'through', 'of', 'is', 'was'}

            # Inject question nouns
            question_words = set(re.findall(r'\b\w+\b', question_text.lower())) - stop_words
            search_words.update(question_words)

            # Inject option nouns
            for opt in options:
                opt_words = set(re.findall(r'\b\w+\b', opt.text.lower())) - stop_words
                search_words.update(opt_words)

            scored_paragraphs = []
            for p_text in paragraphs:
                p_lower = p_text.lower()
                score = sum(1 for word in search_words if word in p_lower)
                scored_paragraphs.append((score, p_text))

            # Sort by highest score first
            scored_paragraphs.sort(key=lambda x: x[0], reverse=True)

            # Extract and group top paragraphs
            context = ""
            for score, p_text in scored_paragraphs[:3]:
                if score > 0:
                    context += p_text + " "

            # Safety Limit: Cap character length to protect inference times from bloated pre-fills
            context = context[:2500].strip()

            # Fallback: if keyword scoring yielded nothing, grab the article leads
            if not context and paragraphs:
                print("   (Keywords not found deep in news text. Falling back to article lead.)")
                context = " ".join(paragraphs[:2])[:2500].strip()

            # ==========================================
            # PHASE 5: SEMANTIC TELEMETRY CHECK
            # ==========================================
            highest_overlap = 0
            best_option = None
            context_words = set(re.findall(r'\b\w+\b', context.lower()))

            for opt in options:
                opt_words = set(re.findall(r'\b\w+\b', opt.text.lower())) - stop_words
                if not opt_words: continue
                overlap_count = len(opt_words.intersection(context_words))
                if overlap_count > highest_overlap:
                    highest_overlap = overlap_count
                    best_option = opt.id

            if highest_overlap >= 2:
                print(f"   (Semantic Match! High overlap detected for Option [{best_option}].)")
            else:
                print("   (Context retrieved, but semantic overlap with options is low.)")

            return context.strip()

        except Exception as e:
            print(f"   (Guardian API Error: {e})")
            return ""

#MATH

For mathematical, theoretical, or abstract algebra problems, standard RAG is insufficient because search engines often lack the depth to solve custom equations.
This is the reason why we are building an Agentic Tool Calling to perform "active" computation.

## AGENTIC TOOL PIPELINE

The system functions as a ReAct (Reason + Act) Agent.
The Mistral model analyzes both the question and the options and selects one of two specialized tools:

  - `execute_secure_python`: For direct computational results (Matrices, Graph Theory, Statistics, Calculus).

  - `search_math_theory`: For qualitative verification (Theorems, logical properties, set theory).

In the case of `execute_secure_python`, the system handles complex math through a multi-library Python stack, ensuring precision where text-based LLMs usually fail:

   - **Numerical Precision**: numpy and scipy for heavy matrix algebra and linear systems.

  - **Abstract Reasoning**: sympy for symbolic math, allowing the engine to handle variables and equations without approximating results.

   - **Structure & Logic**: networkx for graph-based problems and itertools for complex combinatorics.

A **robus result validator** is needed since math outputs and multiple-choice options are often formatted differently (e.g., the code outputs `0.6666` and the option is `2/3`). The normalization layer:

   - **Tolerance-Based Matcher**: A float evaluator (`math.isclose`) that mathematically verifies equivalence rather than relying on string matching.

   - **JSON Auto-Healer**: A robust parser that intercepts and cleans up model-generated JSON, successfully handling common LLM formatting errors (like injected markdown blocks) before the execution tool crashes.

If the model misclassifies a question as a calculation one instead of a theoretical one, the system is able to switch to the `search_math_theory` crawler, which performs a deep-scrape of technical documentation to verify the properties of the set or operation.

Let's dowload the required python libraries.

In [ ]:
pip install torch transformers accelerate requests beautifulsoup4 sympy scipy numpy networkx

In [ ]:
import itertools
import math
import json
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import numpy as np
import networkx as nx
import sympy as sp
from scipy import stats

To enforce the ReAct architecture, we define the tool schema using standard function-calling JSON format. This strict prompt engineering—including explicit "BANS" and "TRIGGERS"—forces Mistral to use specific libraries (`numpy`, `sympy`, `networkx`) depending on the mathematical sub-domain it detects.

In [ ]:
math_router_tools = [
    {
        "type": "function",
        "function": {
            "name": "execute_secure_python",
            "description": """Calculate numerical answers.
BANS: DO NOT use if multiple-choice options contain text, sentences, Roman numerals (e.g., 'I only'), or comparisons.
TRIGGERS:
- [LINEAR ALGEBRA/MATRICES]: Use numpy (import numpy as np). np.linalg.eig, np.linalg.det.
- [GRAPH THEORY]: Use networkx (import networkx as nx). nx.eulerian_circuit, nx.shortest_path.
- [ARITHMETIC/COMBINATORICS]: Use pure Python, math, or itertools.
- [STATISTICS]: Use scipy.stats (imported as stats).
- [ALGEBRA/CALCULUS]: Use sympy (imported as sp). NO sp.Function().""",
            "parameters": {
                "type": "object",
                "properties": {
                    "python_code": {
                        "type": "string",
                        "description": "Python script ending with print(result)."
                    }
                },
                "required": ["python_code"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_math_theory",
            "description": "Verify theorems, logic, abstract matrices/sets, or True/False concepts.",
            "parameters": {
                "type": "object",
                "properties": {
                    "search_query": {
                        "type": "string",
                        "description": "Max 4-5 keywords. No sentences. If multiple statements exist, search ONLY Statement 1."
                    }
                },
                "required": ["search_query"]
            }
        }
    }
]

When the` search_math_theory` tool is triggered, `get_agentic_math_context` acts as an intelligent, two-stage adaptive web crawler:

- **Snippet Evaluation** (The Option-Magnet): It performs an initial search and evaluates the surface-level snippets using an "Option-Magnet" heuristic. If these snippets contain enough high-signal overlap with the multiple-choice options to solve the question, the crawler terminates early to minimize latency.

- **The Deep Scrape**: If the surface snippets yield low-confidence scores, the system dynamically triggers a deep scraper. It navigates to the top technical or academic URLs, parses the raw HTML, and extracts high-quality, long-form paragraphs to ensure the model has sufficient theoretical context.

In [ ]:
def get_agentic_math_context(query, question_text, options, max_results=3):
    """
    Two-Stage Crawler:
    1. Checks DDG snippets.
    2. If snippets lack relevance to the options, it deep-scrapes the top websites.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'
    }

    search_url = "https://html.duckduckgo.com/html/"
    search_params = {'q': query}

    try:
        # --- 1. Fetch Search Results ---
        resp = requests.get(search_url, params=search_params, headers=headers, timeout=5)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')

        results = soup.find_all('div', class_='result', limit=max_results)
        if not results:
            print("   (DDG returned no math theory results.)")
            return ""

        valid_links = []
        snippets_pool = []

        for res in results:
            url_elem = res.find('a', class_='result__url')
            snippet_elem = res.find('a', class_='result__snippet')

            if snippet_elem:
                snippets_pool.append(snippet_elem.text.strip())

            if url_elem:
                link = url_elem['href']
                if not any(bad in link for bad in ['duckduckgo.com', '.pdf']):
                    valid_links.append(link)

        # --- 2. The Snippet Heuristic Check ---
        stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'with', 'by', 'of', 'is', 'was', 'are'}
        target_words = set()

        # Build a magnet list of words from the multiple-choice options
        for opt in options:
            target_words.update(set(re.findall(r'\b\w+\b', opt.text.lower())) - stop_words)

        # Score the snippets based on how many option words they contain
        highest_snippet_score = 0
        for snippet in snippets_pool:
            score = sum(1 for word in target_words if word in snippet.lower())
            if score > highest_snippet_score:
                highest_snippet_score = score

        # --- 3. The Routing Decision ---
        # 1. ALWAYS perform the deep scrape if we want "Theory" accuracy
        # 2. Use snippets only as a fallback or to guide the choice of websites
        if highest_snippet_score >= 3:
             print("   (Snippet confidence high. Using snippets.)")
             return " | ".join(snippets_pool)

        # If score is low (1-2), DO NOT STOP. Proceed to Deep Scrape.
        print("   (Snippets are low-confidence. Deploying deep-crawler...)")

        # --- 4. Deep Scrape (Fallback if Snippets are Starved) ---
        print("   (Snippets are vague. Deploying deep-crawler to read the actual websites...)")
        valid_links = valid_links[:2] # Only read top 2 to preserve speed
        stitched_context = ""

        for url in valid_links:
            domain = urlparse(url).netloc
            print(f"   (Deep Scraping Math Theory: {domain})")

            try:
                page_resp = requests.get(url, headers=headers, timeout=4)
                if page_resp.status_code == 200:
                    page_soup = BeautifulSoup(page_resp.text, 'html.parser')
                    paragraphs = page_soup.find_all('p')

                    chunks = []
                    for p in paragraphs:
                        text = p.get_text().strip()
                        if len(text) > 50: # Ignore UI junk
                            chunks.append(re.sub(r'\s+', ' ', text))
                        if len(chunks) >= 4: # Grab enough for context
                            break

                    if chunks:
                        stitched_context += f"From {domain}: " + " ".join(chunks) + " | "
            except Exception:
                continue

        # Return deep context, or fallback to snippets if scraping was blocked
        return stitched_context.strip() if stitched_context.strip() else " | ".join(snippets_pool)

    except Exception as e:
        print(f"   (Agentic Search Failed: {e})")
        return ""

`execute_tool_call` function serves as the **Execution Kernel and Validation Layer** of the math engine engine. It translates the LLM's raw intent into verified, actionable results by bridging the gap between computational output and multiple-choice options.

Core responsibilities include:

* **JSON Resilience**: features an "Auto-Healer" that recovers and parses tool-call outputs even when the LLM generates malformed or chatty JSON.
* **Secure Execution**: safely executes Python code generated by the model in a sandboxed environment, capturing capturing the standard output (`stdout`) for evaluation.
* **Semantic Mapping**: intelligently compares computed results to the game options using a "Bulletproof Matcher." This handles discrepancies in numerical precision (floats vs. fractions) and structural formats (tuples or ratios), ensuring the code's output maps correctly to the intended Option ID.
* **Agentic Routing**: seamlessly switches between computational tasks (`execute_secure_python`) and conceptual research (`search_math_theory`).

In [ ]:
def execute_tool_call(tool_json_str, question_text, options, model, tokenizer, tracker):

    try:
        # --- 1. THE JSON AUTO-HEALER ---
        try:
            tool_call = json.loads(tool_json_str)
        except json.JSONDecodeError:
            try:
                tool_call = ast.literal_eval(tool_json_str)
            except Exception as e:
                print(f"   (Auto-Healer Failed. Raw string: {tool_json_str})")
                return 0

        if isinstance(tool_call, list) and len(tool_call) > 0:
            tool_call = tool_call[0]

        # --- 2. THE BYPASS CATCHER ---
        if not isinstance(tool_call, dict):
            print(f"   (Mistral bypassed tools and answered directly: {tool_call})")
            clean_val = str(tool_call).replace('[', '').replace(']', '').strip()
            for opt in options:
                if clean_val == str(opt.id) or clean_val.lower() == opt.text.lower().strip():
                    return opt.id
            return 0

        tool_name = tool_call.get("name")
        args = tool_call.get("arguments", {})

        # --- 3. PATH A: PYTHON CALCULATOR ---
        if tool_name == "execute_secure_python":
            print(f"   (Tool Triggered: {tool_name})")
            python_code = args.get("python_code", "")

            # --- THE MARKDOWN STRIPPER ---
            python_code = python_code.strip()
            if python_code.startswith("```python"):
                python_code = python_code[9:]
            elif python_code.startswith("```"):
                python_code = python_code[3:]
            if python_code.endswith("```"):
                python_code = python_code[:-3]
            python_code = python_code.strip()

            print(f"   (Generated Code):\n{python_code}")

            output_buffer = io.StringIO()
            try:
                with contextlib.redirect_stdout(output_buffer):
                    exec(python_code, globals())
                result = output_buffer.getvalue().strip()
                print(f"   (Python Output: '{result}')")

                if not result: raise ValueError("Empty output.")

                clean_result_str = result.lower().replace('[', '').replace(']', '').strip()

                for opt in options:
                    opt_text_clean = opt.text.lower().strip()

                    # 1. Exact text match or Substring match
                    if clean_result_str and (clean_result_str == opt_text_clean or bool(re.search(rf'\b{re.escape(clean_result_str)}\b', opt_text_clean))):
                        return opt.id

                    # 2. The Bulletproof Float Matcher
                    if is_float_match(result, opt.text):
                        return opt.id

                    # 3. Handle Fraction / Ratio Evaluations (e.g. output '2/3' matches option '0.6667')
                    if '/' in result and not '/' in opt.text:
                        try:
                            num, denom = map(float, result.split('/'))
                            if math.isclose(num/denom, float(opt.text), rel_tol=1e-3): return opt.id
                        except: pass

                    # 4. Tuple Fallback
                    if ',' in result and '(' in result:
                        parts = [p.strip() for p in result.replace('(', '').replace(')', '').split(',')]
                        if len(parts) == 2 and parts[0] in opt_text_clean and parts[1] in opt_text_clean: return opt.id

                return 0
            except Exception as e:
                print(f"   (Code Execution Failed: {e})")
                return 0

        # --- 4. PATH B: THEORY RAG AGENT (Fixed to use your math evaluator) ---
        elif tool_name == "search_math_theory":
            print(f"   (Tool Triggered: {tool_name})")
            query = args.get("search_query", "")
            print(f"   (Search Query: '{query}')")

            # Pass question_text and options
            t0 = time.time()
            context = get_agentic_math_context(query, question_text, options)
            tracker["search"] += (time.time() - t0)

            if context: print(f"   Web Context Loaded: {context[:120]}...")
            else: print("   Web Search failed to find context.")

            print("   Asking Mistral for final specialized math evaluation...")
            t1 = time.time()
            ans_id = ask_llm_with_rag_math(question_text, options, context, model, tokenizer)
            tracker["reasoning"] += (time.time() - t1)
            return ans_id

        else:
            print(f"   (Unknown tool: {tool_name})")
            return 0

    except Exception as e:
        print(f"   (Master Engine Failed: {e})")
        return 0

The `is_float_match` function acts as a **mathematical normalization layer** that ensures numerical consistency between the engine's generated output and the multiple-choice options.

* It resolves representation mismatches—such as comparing percentages, fractions, or strings to floating-point numbers—that standard string matching cannot handle.
* It strips formatting characters, uses `sympify` to symbolically evaluate expressions (e.g., converting `1/3` to `0.3333`), and performs a tolerance-based comparison (`math.isclose`) to confirm numerical equivalence despite minor rounding differences.

In [ ]:
def is_float_match(mistral_output, option_text, tolerance=1e-4):
    def extract_float(s):
        # Strip common formatting structures to isolate raw math characters
        s = str(s).replace('$', '').replace('%', '').replace(',', '').strip()
        try:
            return float(sympify(s).evalf())
        except:
            pass

        match = re.search(r'-?\d*\.?\d+(?:[eE][-+]?\d+)?', s)
        if not match:
            return None
        try:
            return float(match.group(0))
        except:
            return None

    val_out = extract_float(mistral_output)
    val_opt = extract_float(option_text)

    if val_out is not None and val_opt is not None:
        return math.isclose(val_out, val_opt, rel_tol=tolerance, abs_tol=1e-8)
    return False

`process_math_question` serves as the **Master Router and Gateway** for all mathematical tasks. By abstracting the complexity of model prompting and JSON parsing away from the main game loop, it ensures every math-related query undergoes a rigorous "Tool vs. Theory" selection process before any code is executed.

Its core mechanics include:

* **Intent Categorization**: It packages the trivia question and options into a strict prompt, utilizing the `math_router_tools` schema to force the Mistral model out of a conversational state and into a deterministic decision-making role.
* **Agentic Routing**: it captures the LLM’s decision, extracts the JSON instructions (via regex), and invokes `execute_tool_call` to perform the actual computation or research.
* **Fallback & Telemetry Management:** If the LLM fails to generate a valid tool call (e.g., if it tries to guess the answer verbally rather than routing it), the function intercepts the failure and forces a safe fallback to prevent a system crash. Additionally, it actively measures and logs "Reasoning" latency to monitor engine performance.


In [ ]:
def process_math_question(question_text, options, model, tokenizer, tracker):
    print("\nMath Router analyzing question...")

    options_text = " ".join([f"[{opt.id}] {opt.text}" for opt in options])
    full_text = f"Question: {question_text}\nOptions:\n{options_text}"

    messages = [
        {"role": "user", "content": full_text}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tools=math_router_tools,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    t0 = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        pad_token_id=tokenizer.eos_token_id
    )
    tracker["reasoning"] += (time.time() - t0)

    input_length = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

    json_match = re.search(r'(\{.*\})|(\[.*\])', response, re.DOTALL)

    if json_match:
        tool_json_str = json_match.group(0)
        try:
            parsed = json.loads(tool_json_str)
            if isinstance(parsed, list) and len(parsed) > 0:
                tool_json_str = json.dumps(parsed[0])
        except Exception:
            pass

        return execute_tool_call(tool_json_str, question_text, options, model, tokenizer, tracker)
    else:
        print(f"   (Mistral failed to generate a tool call. Raw output: {response[:50]}...)")
        print("   (Falling back to Option 0)")
        return 0

`ask_llm_with_rag_math` is the **Final Reasoning Layer** for conceptual math problems as it functions as an "expert auditor". It bridges the gap between raw web context and final selection by forcing the LLM to perform a structured mathematical verification.

Core mechanics include:

* **Contextual Evaluation**: it parses raw, unstructured text extracted from technical websites and forces the LLM to evaluate it against a strict mathematical checklist (e.g., checking for commutativity or finiteness).
* **Structured Prompting**: by injecting a numbered instruction set (1. Identify structure, 2. Identify operation, 3. Cross-reference), it forces "Chain-of-Thought" reasoning even when the final output is heavily constrained.
* **Strict Output Formatting**: it implements a "Critical Requirement" prompt constraint to ensure the output is exclusively a single integer, preventing the model from adding conversational filler that could break the pipeline.

It is implemented:

* **To Solve the "Interpretation Gap":** Mathematical theory is often described in long-form proofs or dense paragraphs; this function ensures the model extracts the *functional properties* needed to map that text to a multiple-choice answer.
* **To Enforce Analytical Rigor:** Instead of asking the model to "find the answer," you are prompting it to verify properties (like Abelian vs. Non-Abelian). This drastically reduces hallucinations because the model must justify the answer against specific structural criteria.
* **Pipeline Consistency:** It serves as the bridge between your "search agent" and the "result ID," ensuring the final step is as deterministic as your code execution path.


In [ ]:
def ask_llm_with_rag_math(question_text, options, context, model, tokenizer):
    options_text = "\n".join([f"  [{opt.id}] {opt.text}" for opt in options])

    math_rag_prompt = (
        "You are an elite abstract algebra and mathematics evaluator. "
        "Your task is to analyze the web context and pick the single best multiple-choice option ID.\n\n"
        f"--- START WEB CONTEXT ---\n{context}\n--- END WEB CONTEXT ---\n\n"
        f"Question: {question_text}\n\n"
        f"Options:\n{options_text}\n\n"
        "INSTRUCTION: Evaluate the web context against the choices using strict mathematical elimination:\n"
        "1. Identify if the mathematical structure/set is finite or infinite.\n"
        "2. Identify if the given operation is commutative (Abelian) or non-commutative (Non-Abelian).\n"
        "3. Cross-reference your answers with the options to select the Option ID that matches ALL verified properties perfectly.\n\n"
        "CRITICAL REQUIREMENT: Output ONLY the raw integer digit representing the correct option ID (e.g., 1). "
        "Do not write conversational text, markdown formatting, or explanations. Just print the digit."
    )

    messages = [
        {"role": "user", "content": math_rag_prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        pad_token_id=tokenizer.eos_token_id
    )

    # Clean input tensor decoding
    input_length = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

    digit_match = re.search(r'\d+', response)
    if digit_match:
        return int(digit_match.group(0))

    return 0

#SPEECH MODE

To implement the Speech Mode, we built a robust, thread-safe transcription pipeline using OpenAI's `Whisper` model.

1. The first major challenge was resource management. Since we are processing audio in parallel, we used `Librosa` to trim dead air concurrently, but we implemented a strict **thread lock** (`threading.Lock()`) around the Whisper inference. This prevents PyTorch from crashing under parallel load by ensuring only one thread uses the AI's core at a time.

2. The second challenge was data cleanliness. Whisper is notorious for acoustic hallucinations. To fix this, we engineered a **7-stage Regex filtering layer**. After transcription, the system automatically strips out non-verbal artifacts, stutters like 'um' and 'uh', and specific acoustic misinterpretations ('prefix cleaner'), which catches garbage words Whisper sometimes hallucinates, like 'Thompson' or 'Topshundi'.

By trimming the audio on the way in, locking the thread during execution, and aggressively filtering the text on the way out, I ensured the speech-to-text pipeline is incredibly fast, stable, and clean.

In [ ]:
# 1. Install the Whisper package and Librosa (for audio trimming)
!pip install -q openai-whisper librosa

# 2. Whisper requires ffmpeg installed on the system to process audio files.
# Colab usually has this, but this guarantees it won't crash.
!sudo apt update && sudo apt install ffmpeg

In [ ]:
import whisper
import librosa
import io
import re
import threading
from concurrent.futures import ThreadPoolExecutor

In [ ]:
# Model load
asr_model = whisper.load_model("base.en")

# Create a thread lock to prevent PyTorch from crashing under parallel load
whisper_lock = threading.Lock()

In [ ]:
def transcribe_audio_bytes(audio_bytes, is_option=False):
    """Takes raw audio bytes, cleans them, and safely pipelines them through Whisper."""
    try:
        # 1. Thread-safe: Downloading and trimming audio happens in parallel
        audio_np, _ = librosa.load(io.BytesIO(audio_bytes), sr=16000)
        audio_np, _ = librosa.effects.trim(audio_np)
        if audio_np.size == 0:
            return ""

        # 2. Locked: Only one thread can use the AI's brain at a time
        with whisper_lock:
            result = asr_model.transcribe(audio_np, fp16=False)

        text = result["text"].strip()

        # 3. Strip Whisper artifacts (coughs, laughs, music notes)
        text = re.sub(r'\[.*?\]|\(.*?\)|\*.*?\*', '', text)

        # 4. Strip verbal fillers
        fillers = [r'\buh+\b', r'\bum+\b', r'\bah+\b', r'\ber+\b', r'\bhm+\b']
        for filler in fillers:
            text = re.sub(filler, '', text, flags=re.IGNORECASE)

        # 5. Destroy spaced-out laughter (ha ha ha) AND attached laughter (hahahaha)
        text = re.sub(r'\b(?:ha|he|hi|ho)(?:[\s,!?]*(?:ha|he|hi|ho))+\b[\s,!?]*', '', text, flags=re.IGNORECASE)

        # 6. Prefix cleaner: Catch "Topshundi", "Thompson", and any variation of "Tops"
        if is_option:
            # Added 'nd' and 's' to the list of intercepted garbage words
            bad_prefixes = r'^(?:Option|Answer|Tops[a-z]*|Top\s+and|Thompson|Pops|nd|s)\s*[A-D]?\s*[:.,-]?\s*'
            text = re.sub(bad_prefixes, '', text, flags=re.IGNORECASE)

        # 7. Clean up weird leftover punctuation (removes those leading commas!)
        text = re.sub(r'\s+', ' ', text)
        return re.sub(r'^[\s,.;:?!-]+|[\s,.;:?!-]+$', '', text).strip()


    except Exception as e:
        print(f"Transcription failed: {e}")
        return ""

#PLAY GAME!

In [ ]:
import time
import warnings
import os
from concurrent.futures import ThreadPoolExecutor

In [ ]:
def play_game_with_ai_and_rag(game, comp_id, model, tokenizer):

    # SYSTEM-LEVEL WARNING SUPPRESSION
    warnings.simplefilter(action='ignore', category=FutureWarning)
    warnings.simplefilter(action='ignore', category=DeprecationWarning)
    os.environ["PYTHONWARNINGS"] = "ignore"

    # RUN-LEVEL TELEMETRY ACCUMULATORS
    total_search_time = 0.0
    total_reasoning_time = 0.0
    total_transcription_time = 0.0
    total_duration = 0.0
    questions_played = 0

    while game.in_progress:

        #  THE SPEECH INTERCEPTOR BLOCK
        if game.mode == "speech":
            transcription_start = time.time()
            q_audio = game.fetch_audio_question()

            audio_tasks = {}
            with ThreadPoolExecutor(max_workers=5) as executor:
                audio_tasks["q"] = executor.submit(transcribe_audio_bytes, q_audio, False)
                for i in range(4):
                    opt_audio = game.fetch_audio_option_next()
                    audio_tasks[f"opt_{i}"] = executor.submit(transcribe_audio_bytes, opt_audio, True)

            # Overwrite the game's text with the new transcriptions
            question = game.current_question
            question.text = audio_tasks["q"].result()
            for i in range(4):
                question.options[i].text = audio_tasks[f"opt_{i}"].result()

            transcription_time = time.time() - transcription_start

            print(f"Transcribed Q: {question.text}")
            for i, opt in enumerate(question.options):
                print(f"   [{i}] {opt.text}")

        else:
            # Normal text mode fallback
            question = game.current_question
            if not question:
                break
            transcription_time = 0.0

        # START THE MASTER TIMER FOR THIS QUESTION
        q_start_time = time.time()

        # Initialize the telemetry tracker for THIS specific question
        q_tracker = {"search": 0.0, "reasoning": 0.0}

        print(f"\n==================================================")
        print(f"LEVEL {game.current_level} | Current Earnings: ${game.earned_amount:,.2f}")
        print(f"==================================================")
        print(f"Q: {question.text}")

        valid_option_ids = []
        for opt in question.options:
            print(f"  [{opt.id}] {opt.text}")
            valid_option_ids.append(opt.id)

        answer_id = None
        context = ""

        # ==========================================
        # GENERAL TRIVIA CATEGORIES (0, 1)
        # ==========================================
        if comp_id in [0, 1]:
            print(f"\nCATEGORY = {comp_id}")

            t0 = time.time()
            context = get_direct_wiki_context(question.text, question.options, model, tokenizer, comp_id)
            q_tracker["search"] += (time.time() - t0)

            if context: print(f"   [ {q_tracker['search']:.2f}s] Search completed. Context extracted.")
            else: print(f"   ⏱️ {q_tracker['search']:.2f}s] Search failed to find context.")

            t1 = time.time()
            answer_id = ask_llm_with_rag(question.text, question.options, context, model, tokenizer, q_tracker["search"])
            q_tracker["reasoning"] += (time.time() - t1)

        # ==========================================
        # ADVANCED TRIVIA CATEGORIES (2, 4)
        # ==========================================
        elif comp_id in [2, 4]:
            print(f"\nCATEGORY = {comp_id}")

            t0 = time.time()
            context = get_direct_wiki_context_detailed(question.text, question.options, model, tokenizer, comp_id)
            q_tracker["search"] += (time.time() - t0)

            if context: print(f"   [ {q_tracker['search']:.2f}s] Search completed. Context extracted.")
            else: print(f"   [ {q_tracker['search']:.2f}s] Search failed to find context.")

            t1 = time.time()
            answer_id = ask_llm_with_rag(question.text, question.options, context, model, tokenizer, q_tracker["search"])
            q_tracker["reasoning"] += (time.time() - t1)

        # ==========================================
        #  NEWS category (5)
        # ==========================================
        elif comp_id == 5:
            print(f"\nCATEGORY = {comp_id}")

            t0 = time.time()
            context = get_guardian_api_context(question.text, question.options, model, tokenizer, comp_id)
            q_tracker["search"] += (time.time() - t0)

            if context: print(f"   [ {q_tracker['search']:.2f}s] Search completed. Context extracted.")
            else: print(f"   [ {q_tracker['search']:.2f}s] Search failed to find context.")

            print("    Asking Mistral for final answer with DuckDuckGo context...")
            t1 = time.time()
            answer_id = ask_llm_with_rag(question.text, question.options, context, model, tokenizer)
            q_tracker["reasoning"] += (time.time() - t1)

        # ==========================================
        #  MATH category (3)
        # ==========================================
        elif comp_id == 3:
            print(f"\nCATEGORY = {comp_id}")

            answer_id = process_math_question(question.text, question.options, model, tokenizer, q_tracker)

        # ==========================================
        # --- THE HARD FALLBACK VALIDATOR ---
        # ==========================================
        if answer_id is None or answer_id not in valid_option_ids:
            safe_id = valid_option_ids[0]
            print(f"   ( FINAL FAILSAFE: RAG did not return a valid option ID. Forcing answer to ID: {safe_id})")
            answer_id = safe_id

        total_q_time = time.time() - q_start_time

        # ==========================================
        # [STEP 2, PART A] ADD METRICS TO ACCUMULATORS
        # ==========================================
        questions_played += 1
        total_search_time += q_tracker["search"]
        total_reasoning_time += q_tracker["reasoning"]
        total_transcription_time += transcription_time
        total_duration += total_q_time

        print(f"AI Selected: {answer_id}")

        # Submit to Game Server
        result = game.answer(answer_id)

        if result.correct:
            print("CORRECT!")
            if result.game_over:
                print(f"\nCONGRATULATIONS! You completed the game!")
                break
        elif result.timed_out:
            print("TIMED OUT! (You exceeded 30 seconds)")
            break
        elif not result.correct:
            print("WRONG ANSWER!")
            break

        # ==========================================
        # ANTI-CRASH RATE LIMITER
        # ==========================================
        print("   (Pausing for 3 seconds to protect the game server...)")
        time.sleep(3)

    # ==========================================
    # [STEP 2, PART B] FINAL GAME SUMMARY & CALCULATIONS
    # ==========================================
    print("\n" + "="*50)
    print("GAME RUN SUMMARY & TELEMETRY")
    print("="*50)
    print(f"Reached Level:  {game.current_level}")
    print(f"Questions Played: {questions_played}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}\n")

    if questions_played > 0:
        avg_search = total_search_time / questions_played
        avg_reasoning = total_reasoning_time / questions_played
        avg_transcription = total_transcription_time / questions_played
        avg_total = avg_search + avg_reasoning + avg_transcription

        print("\n--- Mean Averages per Question ---")
        print(f"Mean Transcription Time: {avg_transcription:.2f} seconds")
        print(f"Mean Search Time:        {avg_search:.2f} seconds")
        print(f"Mean Reasoning Time:     {avg_reasoning:.2f} seconds")
        print(f"Mean Total Time:         {avg_total:.2f} seconds")
    print("==================================================")

In [ ]:
# --- START THE GAME ---
print("\n=== GAME START ===")
comp_id = 5
# Specify the game mode
game = client.game.start(competition_id=comp_id, mode="speech")

play_game_with_ai_and_rag(game, comp_id, model, tokenizer)